In [ ]:
# https://huggingface.co/google/siglip2-so400m-patch16-naflex
import requests
import torch
from PIL import Image
import matplotlib.pyplot as plt

from transformers import AutoModel, AutoProcessor

# google/siglip2-so400m-patch16-naflex: Slower and more precise
# google/siglip2-base-patch16-naflex: Faster and less precise
MODEL_NAME = "google/siglip2-base-patch16-naflex"
device = "cuda" if torch.cuda.is_available() else "cpu"

model = AutoModel.from_pretrained(MODEL_NAME).to(device).eval()
processor = AutoProcessor.from_pretrained(MODEL_NAME)

url = "https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/pipeline-cat-chonk.jpeg"
image = Image.open(requests.get(url, stream=True).raw)
candidate_labels = ["a Pallas cat", "a lion", "a Siberian tiger"]
texts = [f'This is a photo of {label}.' for label in candidate_labels]

# default value for `max_num_patches` is 256, but you can increase resulted image resolution providing higher values e.g. `max_num_patches=512`
inputs = processor(
    text=texts,
    images=image,
    padding="max_length",
    max_length=64,
    truncation=True,
    max_num_patches=256, 
    return_tensors="pt"
).to(device)

with torch.no_grad():
    outputs = model(**inputs)

logits_per_image = outputs.logits_per_image
sigmoid_scores = torch.sigmoid(logits_per_image[0])

for label, score in zip(candidate_labels, sigmoid_scores):
    print(
        f"{label:20s} "
        f"sigmoid={score.item():.4%}"  # スコアの合計は100%にならないので注意
    )

plt.imshow(image)
plt.axis('off')
plt.show()